# 02 - Tokenization & Regular Expressions (Task 3)
**Owner:** Omais Rizvi

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/sample_raw.csv")
print(df.shape)

## 1. Three tokenizers compared
1. **Our regex tokenizer** (`\w+|[^\w\s]`)
2. **NLTK** `wordpunct_tokenize`
3. **spaCy** blank Urdu model (`spacy.blank('ur')`, rule-based Urdu tokenizer)

In [ ]:
import nltk, spacy
from nltk.tokenize import wordpunct_tokenize
nlp = spacy.blank("ur")

def regex_tok(s): return re.findall(r"\w+|[^\w\s]", s)
def nltk_tok(s):  return wordpunct_tokenize(s)
def spacy_tok(s): return [t.text for t in nlp(s)]

sentences = [
    # 1-2: real text from our dataset (no punctuation)
    df["News Text"].iloc[0][:120],
    df["News Text"].iloc[1][:120],
    # 3-4: raw-style text WITH punctuation (what a new article would look like)
    "اسلام آباد: حکومت نے 7.67 فیصد اضافہ کیا، کیا یہ درست ہے؟",
    "آئی ایم ایف کا وفد (IMF) پیر کو پاکستان پہنچے گا۔ مزید پڑھیں: www.dawnnews.tv",
]
rows = []
for s in sentences:
    a, b, c = regex_tok(s), nltk_tok(s), spacy_tok(s)
    rows.append({"sentence": s[:50] + "...", "regex": len(a), "nltk": len(b), "spacy": len(c)})
    print("TEXT :", s); print("REGEX:", a); print("NLTK :", b); print("SPACY:", c); print()
pd.DataFrame(rows)

**Observations (edit in your own words):**
- On our dataset text (no punctuation) all three give almost the same tokens, because Urdu words are separated by spaces.
- On raw text, `7.67` is split into `7 . 67` by regex and NLTK, but spaCy keeps it as one token.
- The URL is broken into many pieces by regex/NLTK; spaCy keeps it as one token.
- We use simple whitespace splitting in the pipeline, because punctuation/URLs are removed before tokenization.

## 2. Regular expressions for our data
Patterns are in `src/regex_patterns.py`.

In [ ]:
import regex_patterns as rp
rp.demo()

In [ ]:
text = df["News Text"]
results = {
    "NUMBER (any number)":            text.str.count(rp.NUMBER.pattern).sum(),
    "NUMBER_WITH_UNIT (20 لاکھ ...)": text.str.count(rp.NUMBER_WITH_UNIT.pattern).sum(),
    "MERGED_FUTURE (گااس ...)":       text.str.count(rp.MERGED_FUTURE.pattern).sum(),
    "LATIN_WORD (English)":           text.str.count(rp.LATIN_WORD.pattern).sum(),
}
print(pd.Series(results))
units = Counter(u for t in text for _, u in rp.NUMBER_WITH_UNIT.findall(t))
print("\nMost common units:", units.most_common())
by_cat = df.groupby("Category")["News Text"].apply(lambda s: s.str.count(rp.NUMBER_WITH_UNIT.pattern).mean())
print("\nAvg number+unit mentions per article by category:"); print(by_cat.round(2))

In [ ]:
ids = df["URL"].str.extract(rp.URL_ARTICLE_ID.pattern)[0]
print("Article IDs from URL:", ids.head().tolist())
dates = df["Date"].astype(str).str.extract(rp.DATE.pattern)
print("Years in sample:"); print(dates[0].value_counts().sort_index())

## 3. Pattern explanations (edit in your own words)
| Pattern | Regex | What / why |
|---|---|---|
| NUMBER | `\d+(?:[.,]\d+)*` | Finds numbers like 20, 1,500, 7.67. Later replaced by `NUM` so every number is not a new word. |
| NUMBER_WITH_UNIT | `(\d+...)\s*(ہزار\|لاکھ\|کروڑ\|ارب\|فیصد...)` | Money/amount phrases. Business news has many of them - useful signal. |
| MERGED_FUTURE | `\b(گا\|گی\|گے)(اس\|ان\|یہ\|وہ...)\b` | Finds words glued together because the full stop was deleted. |
| URL_ARTICLE_ID | `/news/(\d+)` | Gets the article id from the URL column (for removing duplicates). |
| DATE | `(\d{4})-(\d{2})-(\d{2})` | Splits the date into year / month / day. |

## 4. Tokenization challenges in our data
1. **No sentence boundaries** - punctuation was removed, so words get glued (`ہےفنانس`, `کرے گااس`). Tokenizer sees it as one word. We fix the `ے`/`ں` cases in `clean.py`.
2. **Multi-word names** - `اسلام آباد`, `آئی ایم ایف` are one thing but become 2-3 tokens (and here the آ is even missing).
3. **Compound / joined words** - `خیبرپختونخوا` is written joined, but similar names may be written with a space; the same word can have two spellings.
4. **Numbers with decimals / slashes** - `7.67` breaks into `7 . 67` with simple tokenizers; in our data the dot and slash were deleted, so `2020-21` became `212020` and `کراچی/لاہور` became `کراچیلاہور`.